# GRIB: Interpolating from hybrid to pressure levels

This notebook demonstrates how to interpolate GRIB fieldlist data on hybrid model levels to pressure levels. Fieldlist support requires the usage of earthkit-data.

In [1]:
import earthkit.data as ekd
import numpy as np

import earthkit.meteo.vertical.fieldlist as vertical

## Hybrid levels

## Getting the data

The input is GRIB data on 137 IFS model levels for a single step. We fetch the file and extract temperature, specific humidity and the surface pressure.

In [2]:
fl = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-meteo/test-data/tq_ml137.grib2",
).to_fieldlist()

# LNSP → surface pressure [Pa]
lnsp = fl.sel({"parameter.variable": "lnsp", "vertical.level": 1})[0]
sp = lnsp.set(values=np.exp(lnsp.values))

t = fl.sel({"parameter.variable": "t"})  # temperature, 137 levels
q = fl.sel({"parameter.variable": "q"})  # specific humidity, 137 levels

t.head()

tq_ml137.grib2:   0%|          | 0.00/388k [00:00<?, ?B/s]

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,1,hybrid,None,regular_ll
1,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,2,hybrid,None,regular_ll
2,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,3,hybrid,None,regular_ll
3,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,4,hybrid,None,regular_ll
4,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,5,hybrid,None,regular_ll


The A and B coefficients defining the hybrid levels are embedded in the GRIB metadata and are inferred automatically.

## Using interpolate_hybrid_to_pressure_levels

In [3]:
target_p = [85000.0, 50000.0]  # Pa

t_res = vertical.interpolate_hybrid_to_pressure_levels(
    t,  # data to interpolate
    target_p,
    sp,
    interpolation="linear",
)
t_res.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,850.0,pressure,None,regular_ll
1,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,500.0,pressure,None,regular_ll


The result is a FieldList with one field per target pressure level. Pressure levels in the output metadata are stored in hPa.

In [9]:
# the first value from the resulting fields
t_res.values[:, 0]

array([279.24659968, 256.84623177])

### Using a subset of levels

It is possible to use only a **subset of the model levels** in the input data. This can significantly speed up the computations and reduce memory usage.

The model level range must be contiguous and include the bottom-most level. The example below uses only the 50 lowest model levels.

In [4]:
# keep only the 50 lowest model levels
t_sub = t.sel({"vertical.level": list(range(88, 138))})

t_res_sub = vertical.interpolate_hybrid_to_pressure_levels(
    t_sub,
    target_p,
    sp,
    interpolation="linear",
)
t_res_sub.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,850.0,pressure,None,regular_ll
1,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,500.0,pressure,None,regular_ll


In [10]:
# the first value from the resulting fields
t_res.values[:, 0]

array([279.24659968, 256.84623177])

### Using aux levels

The pressure on the lowest model level defines the maximum target pressure for interpolation for each grid point. We can check this limiting pressure.

In [16]:
# the first 4 values of pressure on the lowest model level (Pa)
p_bottom = vertical.pressure_on_hybrid_levels(sp, levels=[137], output="full")
p_bottom[0].values[:4]

array([101170.51199572, 100410.77168025,  98254.02652169,  97513.21165167])

As a consequence, interpolation to 100000 Pa results in NaN for e.g. point 2 and. We can verify this:

In [17]:
# 100000 Pa is below the pressure of the lowest model level in grid points 2 and 3 → NaN
target_p_low = [100000.0]  # Pa

t_res_nan = vertical.interpolate_hybrid_to_pressure_levels(
    t,
    target_p_low,
    sp,
    interpolation="linear",
)
t_res_nan.values[:, :4]

array([[284.0277523 , 287.39509546,          nan,          nan]])

To overcome this problem we can supply auxiliary data via the `aux_bottom_*` kwargs. As a demonstration, we simply define an auxiliary pressure level with a constant temperature.

In [7]:
t_res_aux = vertical.interpolate_hybrid_to_pressure_levels(
    t,
    target_p_low,
    sp,
    aux_bottom_p=100100.0,  # Pa — pressure of the auxiliary level
    aux_bottom_data=300.0,  # K  — temperature at that level
    interpolation="linear",
)
t_res_aux.to_numpy()[:, :2, :2]

array([[[289.43196211, 292.33333564],
        [298.91595963, 299.00674841]]])

## Writing to GRIB

In [8]:
t_res.to_target("file", "_hybrid_to_pl.grib")

# read back and verify
ekd.from_source("file", "_hybrid_to_pl.grib").to_fieldlist().ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,8,pressure,None,regular_ll
1,t,2019-06-02 12:00:00,2019-06-02 12:00:00,0 days,5,pressure,None,regular_ll
